# UK Online Retail Sales Analysis — Visualizations

This notebook connects directly to the `retail_analysis` PostgreSQL database
and produces the key charts referenced in `README.md`. Keeping this separate
from `data_cleaning.ipynb` mirrors a typical real-world workflow: data
cleaning/loading is a one-off pipeline step, while visualization/reporting
is repeatable and reruns against the database directly.

**Charts produced:**
1. Monthly revenue trend (seasonality)
2. Cumulative revenue concentration by customer (Pareto)
3. Customer segmentation by order frequency (retention)

Each chart is saved as a PNG for embedding in the README.


## 1. Connect to PostgreSQL

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine

# Adjust username if your local PostgreSQL role differs
engine = create_engine("postgresql://yunusajib@localhost:5432/retail_analysis")

# Consistent styling for all charts in this notebook
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print("Connected to retail_analysis database.")


## 2. Chart 1 — Monthly Revenue Trend

Pulls monthly aggregates directly via SQL (rather than pandas groupby) to keep the notebook's numbers consistent with `analysis_queries.sql` Query 1.

In [ ]:
query_monthly_revenue = """
    SELECT
        DATE_TRUNC('month', invoice_date) AS month,
        ROUND(SUM(total_price), 2) AS total_revenue
    FROM clean_sales
    GROUP BY DATE_TRUNC('month', invoice_date)
    ORDER BY month;
"""

monthly_revenue = pd.read_sql(query_monthly_revenue, engine)
monthly_revenue['month'] = pd.to_datetime(monthly_revenue['month'])

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly_revenue['month'], monthly_revenue['total_revenue'],
        marker='o', color='#2563eb', linewidth=2)
ax.set_title('Monthly Revenue Trend (Dec 2009 - Dec 2011)', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (£)')
ax.grid(alpha=0.3)
ax.yaxis.set_major_formatter(lambda x, pos: f'£{x/1000:,.0f}k')
fig.tight_layout()
fig.savefig('chart_monthly_revenue.png', dpi=150)
plt.show()

print("Saved: chart_monthly_revenue.png")


## 3. Chart 2 — Customer Revenue Concentration (Pareto)

Visualizes what % of total revenue is generated by the top N customers, supporting the finding in `analysis_queries.sql` Query 2b.

In [ ]:
query_customer_revenue = """
    SELECT
        customer_id,
        SUM(total_price) AS total_spent
    FROM clean_sales
    GROUP BY customer_id
    ORDER BY total_spent DESC;
"""

customer_revenue = pd.read_sql(query_customer_revenue, engine)
customer_revenue['cumulative_pct'] = (
    customer_revenue['total_spent'].cumsum() / customer_revenue['total_spent'].sum() * 100
)
customer_revenue['customer_rank'] = range(1, len(customer_revenue) + 1)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(customer_revenue['customer_rank'], customer_revenue['cumulative_pct'],
        color='#dc2626', linewidth=2)

# Highlight the top-100-customers reference point
top_100_pct = customer_revenue.loc[99, 'cumulative_pct']
ax.axhline(y=top_100_pct, color='gray', linestyle='--', alpha=0.6)
ax.axvline(x=100, color='gray', linestyle='--', alpha=0.6)
ax.annotate(f'Top 100 customers = {top_100_pct:.1f}% of revenue',
            xy=(100, top_100_pct), xytext=(400, top_100_pct - 8),
            fontsize=10, color='#374151')

ax.set_title('Cumulative Revenue by Customer Rank', fontsize=13, fontweight='bold')
ax.set_xlabel('Customer Rank (by total spend)')
ax.set_ylabel('Cumulative % of Total Revenue')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig('chart_revenue_concentration.png', dpi=150)
plt.show()

print(f"Saved: chart_revenue_concentration.png")
print(f"Top 100 customers generate {top_100_pct:.1f}% of total revenue")


## 4. Chart 3 — Customer Segmentation by Order Frequency

Supports the retention finding in `analysis_queries.sql` Query 3 — visualizes the split between one-time, repeat, and loyal customers.

In [ ]:
query_segments = """
    WITH customer_orders AS (
        SELECT
            customer_id,
            COUNT(DISTINCT invoice) AS num_orders
        FROM clean_sales
        GROUP BY customer_id
    )
    SELECT
        CASE
            WHEN num_orders = 1 THEN 'One-time\n(1 order)'
            WHEN num_orders BETWEEN 2 AND 5 THEN 'Repeat\n(2-5 orders)'
            ELSE 'Loyal\n(6+ orders)'
        END AS customer_segment,
        COUNT(*) AS num_customers
    FROM customer_orders
    GROUP BY customer_segment;
"""

segments = pd.read_sql(query_segments, engine)
# Order segments logically rather than alphabetically
segment_order = ['One-time\n(1 order)', 'Repeat\n(2-5 orders)', 'Loyal\n(6+ orders)']
segments['customer_segment'] = pd.Categorical(segments['customer_segment'], categories=segment_order, ordered=True)
segments = segments.sort_values('customer_segment')

colors = ['#94a3b8', '#60a5fa', '#2563eb']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(segments['customer_segment'], segments['num_customers'], color=colors)

# Label each bar with count and percentage
total = segments['num_customers'].sum()
for bar, count in zip(bars, segments['num_customers']):
    pct = count / total * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{count:,}\n({pct:.1f}%)', ha='center', fontsize=10)

ax.set_title('Customer Segmentation by Purchase Frequency', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Customers')
ax.set_xlabel('')
fig.tight_layout()
fig.savefig('chart_customer_segments.png', dpi=150)
plt.show()

print("Saved: chart_customer_segments.png")


## 5. Summary

Three charts have been saved to the project folder:
- `chart_monthly_revenue.png`
- `chart_revenue_concentration.png`
- `chart_customer_segments.png`

Next step: embed these in `README.md` using standard markdown image syntax:

```markdown
![Monthly Revenue Trend](chart_monthly_revenue.png)
```

Then commit and push the updated README and PNG files to GitHub.